<a href="https://colab.research.google.com/github/tiwariaxay/PDF-RAG-CHATBOT/blob/chatbot/Widows_ChatBot_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers sentence-transformers faiss-cpu pypdf torch accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 22.2 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)

Saving Windows 11 User Guide.pdf to Windows 11 User Guide.pdf
Uploaded file: Windows 11 User Guide.pdf


In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

raw_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text += text + "\n"

print("Total characters:", len(raw_text))
print(raw_text[:1000])

Total characters: 9474
User Guide
1. Turn on the computer, and press the power button      for booting.
When turning on the computer for the ﬁrst time, please connect the power adapter ﬁrst, 
then the computer will turn on automatically. As the screen lights up, enter the boot 
setup interface. 
When the computer is turned off or goes to sleep, it can be turned on or wake up by 
short pressing the power button until the keyboard indicator lights up.
While using, click      >      to put the computer to sleep, shut down or restart. 
Forced shutdown: press and hold the power button for 10 seconds or longer to force 
shutdown. The forced shutdown will result in the loss of any unsaved data, please 
use it with caution.
2. When booting up for the ﬁrst time, the system will enter the initialization 
process which may take some time. Please wait patiently until the language 
selection interface appears.
01
System Boot Up & Initialization
01
01
4. Select the correct country (region), and clic

In [4]:
chunk_size = 500
chunks = []

for i in range(0, len(raw_text), chunk_size):
    chunks.append(raw_text[i:i+chunk_size])

print("Total chunks:", len(chunks))

Total chunks: 19


In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)

print("Embedding created:", len(embeddings))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding created: 19


In [6]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

Vectors stored: 19


In [12]:
query = "pls provide After-sales service contacts?"

query_embedding = embedding_model.encode([query])

In [8]:
k = 3

distances, indices = index.search(np.array(query_embedding), k)

retrieved_chunks = [chunks[i] for i in indices[0]]

for c in retrieved_chunks:
    print(c)
    print("------")

e right 
ﬁrmware, e.g. G1F1:
After entering, you can see the download link list of the ﬁrmware:
26

After-sales service contacts:
If you encounter unsolvable product issues, please email customersupport@teclast.com
The email must contain the following 3 things, otherwise the after-sales service will 
not be provided.
1. The name of the platform where you purchased the product (if it is a country-based 
platform, please provide the name of the country);
2. The product model, four-digit ID number,
------
the Start menu. It can help you master various 
features of Windows 11 quickly.
06
1. Connect to a network
Click [Mark 1] in the taskbar in the bottom right of the desktop, and click the 
arrow in the [Mark 2] in the pop-up menu;
Scroll to ﬁnd the WiFi to be connected and click [Connect];
Windows Settings
07
Enter the network security key, and click [Next]; (no security key for open 
wireless networks)
When the password and other information are veriﬁed, the network is connected.
08
When